# 20 · Training on Databricks: MLflow and the Unity Catalog registry

**Who this is for:** you, after notebook 06 (the first model, on your laptop) and notebook 13 (Gold).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Gold must exist in the catalog you pick (`heavden_dev` after a pipeline run).

**By the end you will be able to:**
- explain what the `train` job does, step by step, and why it's the same code as notebook 06
- find a training run in the MLflow experiment and read its metrics against NEWS2
- explain **aliases**: `@champion` is what production uses, `@challenger` is a model that passed the gates and waits for a person to promote it
- load a registered model by alias and score patient-hours with it
- say why the model carries its own code and pinned library versions

The job is `databricks/resources/train.job.yml`; the logic is `heavden.ml.training`, tested locally (`tests/test_training.py`), also on Databricks' own library versions (CI job `ml-on-databricks-libraries`).

## 1. Setup

The job installs LightGBM itself; a notebook needs it installed once per session.

In [ ]:
%pip install lightgbm==4.6.0 --quiet

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

import mlflow

from heavden.ml import train, training

dbutils.widgets.text("catalog", "heavden_dev")
CATALOG = dbutils.widgets.get("catalog")
MODEL = f"{CATALOG}.ml.deterioration_risk"
mlflow.set_registry_uri("databricks-uc")
print(repo_root, CATALOG, MODEL)

## 2. The training data and the time split

The job reads `gold.training_set` (only hours whose label is final) into pandas: under 100,000 rows, so one machine is plenty. At hospital scale you'd train on Spark or sample.

It splits **by time**, never at random: hours of the same stay are strongly alike, so a random split would put near-copies of test rows in training and flatter the model (Phase 0: AUPRC 0.40 became 0.51). 6-hour gaps keep a label window from straddling two blocks.

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
table = training._utc(
    spark.read.table(f"{CATALOG}.gold.training_set").toPandas(), ("prediction_ts", "label_known_at")
)
days = (1, 0.5) if CATALOG != "heavden_prod" else (9, 2)  # dev/staging hold 2 days, prod 14
tr, va, te = train.time_split(table, train.Split(*days))
for name, part in [("train", tr), ("validation", va), ("test", te)]:
    print(
        f"{name:10} {len(part):>7,} hours  {int(part.label.sum()):>4} positive  "
        f"{part.prediction_ts.min()} .. {part.prediction_ts.max()}"
    )

## 3. The runs in the MLflow experiment

Every training run logs one MLflow run per candidate (logistic regression, LightGBM) with its parameters, validation metrics and NEWS2's on the same hours (`news2_valid_*`). Only the **chosen** one is scored on test (`test_*`): test is looked at once, so it stays an honest estimate.

The experiment is in the deploying identity's home: yours for dev, the robot's for staging and prod (Workspace → Users → its ID → `heavden-deterioration-risk`).

In [ ]:
user = spark.sql("SELECT current_user()").first()[0]
experiment = f"/Workspace/Users/{user}/[dev {user.split('@')[0]}] heavden-deterioration-risk"
runs = mlflow.search_runs(experiment_names=[experiment], order_by=["start_time DESC"])
cols = [
    "tags.mlflow.runName",
    "tags.chosen",
    "tags.passed_gates",
    "metrics.valid_auprc",
    "metrics.news2_valid_auprc",
    "metrics.valid_escalations_flagged",
    "metrics.news2_valid_escalations_flagged",
    "metrics.test_auprc",
    "start_time",
]
display(runs[[c for c in cols if c in runs]])

If the experiment isn't found, open **Experiments** in the left menu: dev mode prefixes resource names with `[dev <you>]`, so the exact name can differ.

## 4. The registry: versions and aliases

Each run registers its chosen model as a **new version** of `<catalog>.ml.deterioration_risk`, so the registry is a complete audit trail. Aliases say which version does what:

| Alias | Meaning | Set by |
|---|---|---|
| `@champion` | the model production scores with; the serving endpoint follows it | the first model that passes the gates; later, a person approving a promotion (Phase 3) |
| `@challenger` | the newest model that passed the gates | the training job |

Swapping `@champion` to another version **is** the deployment, and swapping it back is the rollback. Model stages (Staging/Production) are deprecated; don't use them.

In [ ]:
client = mlflow.MlflowClient()
for v in client.search_model_versions(f"name = '{MODEL}'"):
    print(f"v{v.version}  aliases={v.aliases}  passed_gates={v.tags.get('passed_gates')}")
    print(f"    {v.description}")

## 5. Load a model by alias and score with it

`models:/<name>@champion` always means "whatever production uses now". In dev and staging, the 2 days of data are usually too few to pass the gates, so there may be no champion: then load a version number instead.

In [ ]:
alias = "champion"
try:
    client.get_model_version_by_alias(MODEL, alias)
    uri = f"models:/{MODEL}@{alias}"
except mlflow.exceptions.MlflowException:
    latest = max(int(v.version) for v in client.search_model_versions(f"name = '{MODEL}'"))
    uri = f"models:/{MODEL}/{latest}"
model = mlflow.sklearn.load_model(uri)
sample = te.head(10).copy()
sample["risk"] = model.predict_proba(sample)[:, 1]
display(sample[["encounter_id", "prediction_ts", "news2_total", "risk", "label"]])

## 6. Why the model carries its own code

The model is our `CalibratedModel` class pickled with cloudpickle, so loading it needs that class. It is logged with `code_paths` (the `heavden` package, copied inside the model) and `pip_requirements` pinned to the exact library versions it was trained with. A scoring job or the serving endpoint can then load it without our wheel, and can't silently load it with a different scikit-learn.

In [ ]:
local = mlflow.artifacts.download_artifacts(uri)
print(open(f"{local}/requirements.txt").read())
print(sorted(p.name for p in __import__("pathlib").Path(local, "code", "heavden").iterdir()))

## 7. Try it yourself

1. In the experiment, compare the two candidates' `valid_auprc` and `valid_escalations_flagged` with NEWS2's. Which one was chosen, and did it pass the gates?
2. Run `training.run_training(table, f"{CATALOG}.ml.deterioration_risk_sandbox", train.Split(*days))` twice (after `mlflow.set_experiment(...)` to a sandbox experiment). Which aliases does each version get, and why does the second never become `@champion`?
3. Score the test block with the champion and with `train.news2_scores`, and compare how many escalations each flags at the alert budget (`heavden.ml.evaluate.evaluate`).
4. Why is test scored only for the chosen model? What would go wrong if we picked the champion on test?

In [ ]:
# Try it yourself: 1


In [ ]:
# Try it yourself: 2


**Pitfall: picking on test.** Choose and tune on validation; look at test once. Otherwise test stops being an honest estimate.

**Pitfall: a model that doesn't match the runtime.** Train and score with the same library versions; that's why the package and the model pin them.

**Next:** batch scoring (`gold.risk_scores`) with `@champion`, then the alerts and KPI tables.